# Regioni

Rendiconti regionali OpenBDAP/FET. La prima cella richiama il codice Python del repo: se l'input non esiste, genera `source-data.json` e materializza la sezione `regioni`.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from bilancio_pubblico.notebook_inputs import load_section, frame, print_input_status

SECTION_ID = "regioni"
REFRESH = False
FORCE_DOWNLOAD = False

regioni = load_section(SECTION_ID, refresh=REFRESH, force=FORCE_DOWNLOAD)
print_input_status(SECTION_ID)

In [ ]:
import matplotlib.pyplot as plt

def plot_bar(data, x, y, title, top=15):
    df = data.dropna(subset=[y]).sort_values(y, ascending=False).head(top).sort_values(y)
    ax = df.plot.barh(x=x, y=y, legend=False, figsize=(9, max(4, len(df) * 0.35)))
    ax.set_title(title)
    ax.set_xlabel(y)
    ax.set_ylabel("")
    plt.tight_layout()
    return ax

## Copertura

Copertura per anno e comparto. Serve a controllare quali regioni sono presenti.

In [ ]:
coverage = frame(regioni["coverage"])
display(coverage)

## Entrate finali per Regione

Le entrate finali sommano i titoli 1-5 e quindi escludono accensione prestiti, anticipazioni e partite di giro.

In [ ]:
rev_agg = frame(regioni["revenue"]["aggregates_by_region"])
entrate_finali = rev_agg[rev_agg["aggregate_id"] == "entrate_finali"]
latest_year = entrate_finali["anno"].max() if not entrate_finali.empty else None
display(entrate_finali[entrate_finali["anno"] == latest_year].sort_values("mld", ascending=False).head(30))
plot_bar(entrate_finali[entrate_finali["anno"] == latest_year], "regione", "mld", f"Entrate finali regionali {latest_year}")

## Composizione delle entrate

Vista per titolo delle entrate regionali. Cambia `REGIONE` per analizzare una regione specifica.

In [ ]:
REGIONE = "Sardegna"
revenue_titles = frame(regioni["revenue"]["by_title"])
latest_title_year = revenue_titles["anno"].max() if not revenue_titles.empty else None
selected = revenue_titles[(revenue_titles["regione"] == REGIONE) & (revenue_titles["anno"] == latest_title_year)]
display(selected[["titolo_code", "titolo", "mld", "euro_per_capita"]].sort_values("titolo_code"))
plot_bar(selected, "titolo", "mld", f"Entrate per titolo - {REGIONE} {latest_title_year}")

## Saldo finale

Il saldo finale è calcolato come entrate finali meno spese finali. È distinto dal saldo totale storico, che resta disponibile nel payload regionale.

In [ ]:
balances = frame(regioni["balances"]["final_by_region"])
latest_balance_year = balances["anno"].max() if not balances.empty else None
display(balances[balances["anno"] == latest_balance_year].sort_values("mld"))
plot_bar(balances[balances["anno"] == latest_balance_year], "regione", "mld", f"Saldo finale regionale {latest_balance_year}")